In [4]:
%pip install -q rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import os, pandas as pd
from rank_bm25 import BM25Okapi

In [14]:
import json

# Carica la mappatura nome_file -> id dal file jsonl
name2id = {}
with open("data_search_e_collection.jsonl", "r", encoding="utf-8") as fin:
    for line in fin:
        obj = json.loads(line)
        # Usa il primo elemento di 'data' per ottenere 'data_filename'
        if 'data' in obj and isinstance(obj['data'], list) and obj['data']:
            data_filename = obj['data'][0].get('data_filename')
            if data_filename:
                name2id[data_filename] = obj['id']

docs, doc_ids = [], []
data_dir = "datasets_90_csv"
if not os.path.isdir(data_dir):
    raise FileNotFoundError(f"Directory not found: {data_dir}")
for f in os.listdir(data_dir):
    if f.endswith(".csv"):
        # Usa l'id dal mapping, altrimenti salta il file
        if f not in name2id:
            continue
        did = name2id[f]
        df = pd.read_csv(os.path.join(data_dir, f))
        text = " ".join(df.columns) + " " + " ".join(df.astype(str).values.flatten())
        tokens = text.lower().split()
        docs.append(tokens)
        doc_ids.append(did)


In [7]:
# 2) Costruzione BM25
bm25 = BM25Okapi(docs)

In [8]:
# 3. Query in linguaggio naturale (modifica secondo il tuo caso)
queries = {
    "DS1-E-0005": "births by month",
    "DS1-E-0016": "trend number police officer 1945",
    "DS1-E-0045": "recent gender birth rate in us",
    "DS1-E-0049": "united states births and deaths per second"
}

In [15]:
results = {}
for qid, qtext in queries.items():
    query_tokens = qtext.lower().split()
    scores = bm25.get_scores(query_tokens)
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    results[qid] = [doc_id for doc_id, _ in ranked[:10]]

In [2]:
%pip install -q trectools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [16]:
import trectools
from collections import defaultdict

# === 4. Esegui il ranking BM25 e salva i risultati in formato TREC ===
results = {}
trec_lines = []

for qid, qtext in queries.items():
    tokens = qtext.lower().split()
    scores = bm25.get_scores(tokens)
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    
    results[qid] = {}
    for rank, (doc_id, score) in enumerate(ranked[:10]):
        results[qid][doc_id] = float(score)
        # formato: <query-id> Q0 <doc-id> <rank> <score> <run-name>
        trec_line = f"{qid} Q0 {doc_id} {rank} {score:.4f} BM25-baseline"
        trec_lines.append(trec_line)

# Scrivi i risultati su file in formato TREC
with open("results_baseline_bm25.txt", "w") as f:
    for line in trec_lines:
        f.write(line + "\n")

In [18]:
from trectools import TrecRun, TrecQrel, TrecEval

# Carica il file di run e il file di qrels
run = TrecRun("results_baseline_bm25.txt")
qrels = TrecQrel("../qrels_filtered_by_result.txt")

results = TrecEval(run, qrels)

# Stampa alcune metriche comuni
print(f"nDCG@10: {results.get_ndcg(depth=10)}")
print(f"nDCG@5: {results.get_ndcg(depth=5)}")
print(f"nDCG@3: {results.get_ndcg(depth=3)}")
print(f"Precision@10: {results.get_precision(depth=10)}")
print(f"Recall: {results.get_recall()}")

nDCG@10: 0.455093745428916
nDCG@5: 0.38725226453547945
nDCG@3: 0.4544804218469287
Precision@10: 0.325
Recall: 0.4883116883116883
